In [ ]:
#version-3.1

# ================================
# SWED Water Segmentation — v3.1
# Adapted for SWED_real dataset (28,224 .npy train pairs)
#
# Key changes from v3:
#   ★ IN_CHANNELS = 14  (added MNDWI as channel 14)
#   ★ compute_mndwi() helper using Green−SWIR1 (band 11)
#   ★ Both NDWI + MNDWI concatenated → better turbid/mountain water
#   ★ All other v3 improvements retained
#
# Why MNDWI helps:
#   NDWI  = (Green − NIR)  / (Green + NIR  + ε)  → works on clear water
#   MNDWI = (Green − SWIR) / (Green + SWIR + ε)  → works on turbid/sediment-laden water
#   Together they cover braided glacial rivers, coastal estuaries, shadowed channels.
# ================================

# ================================
# IMPORTS
# ================================
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
import segmentation_models_pytorch as smp
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ================================
# CONFIG  ← main knobs to tune
# ================================
BASE_DIR     = "/home/jupyter-228w1a1286/dinesh-major/SWED_real/SWED/train"
TEST_DIR     = "/home/jupyter-228w1a1286/dinesh-major/SWED_real/SWED/test"
IMG_DIR      = os.path.join(BASE_DIR, "images")
MASK_DIR     = os.path.join(BASE_DIR, "labels")
SAVE_PATH    = "/home/jupyter-228w1a1286/dinesh-major/v3/segformer_v3_1_best.pth"

# ── Batch & accumulation ──────────────────────────────────────────────────────
BATCH_SIZE        = 16      # fits on 16 GB GPU; drop to 8 if OOM
ACCUM_STEPS       = 2       # effective batch = BATCH_SIZE × ACCUM_STEPS = 32

# ── Training duration ─────────────────────────────────────────────────────────
EPOCHS            = 30

# ── Learning rate ─────────────────────────────────────────────────────────────
LR                = 1e-4
LR_MIN            = 5e-6
WARMUP_EPOCHS     = 3

# ── Architecture ──────────────────────────────────────────────────────────────
ENCODER           = "mit_b4"
ARCH              = "segformer"

# ── Data ──────────────────────────────────────────────────────────────────────
VAL_SPLIT         = 0.1
IN_CHANNELS       = 14      # ★ 12 spectral + NDWI + MNDWI
IMG_SIZE          = 256
NUM_WORKERS       = 8

# ── Misc ──────────────────────────────────────────────────────────────────────
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
SEED              = 42
torch.manual_seed(SEED); np.random.seed(SEED)

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
print(f"Device : {DEVICE}")
print(f"Config : encoder={ENCODER}  batch={BATCH_SIZE}×{ACCUM_STEPS}  "
      f"lr={LR}  epochs={EPOCHS}  in_channels={IN_CHANNELS}")


# ================================
# QUICK SHAPE CHECK
# ================================
def check_shapes(img_dir, mask_dir, n=3):
    img_files  = sorted([f for f in os.listdir(img_dir)  if f.endswith(".npy")])[:n]
    mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith(".npy")])[:n]
    print("=== Shape check ===")
    for f in img_files:
        arr = np.load(os.path.join(img_dir, f))
        print(f"  IMG   {f[:50]}  shape={arr.shape}  dtype={arr.dtype}")
    for f in mask_files:
        arr = np.load(os.path.join(mask_dir, f))
        print(f"  MASK  {f[:50]}  shape={arr.shape}  unique={np.unique(arr)}")
    print("===================\n")

check_shapes(IMG_DIR, MASK_DIR)


# ================================
# SPECTRAL INDEX HELPERS
#
# Sentinel-2 band layout (0-indexed, 12 bands stored in .npy):
#   0:B2  1:B3(Green?)  2:B3(Green)  3:B4  4:B5  5:B6
#   6:B7  7:B8(NIR)     8:B8A        9:B9  10:B11(SWIR1) 11:B12(SWIR2)
#
# NDWI  = (Green − NIR)  / (Green + NIR  + ε)   bands: 2, 7
# MNDWI = (Green − SWIR) / (Green + SWIR + ε)   bands: 2, 10
#
# ★ MNDWI advantage: SWIR penetrates water much less than NIR,
#   so turbid/sediment-laden water (glacial rivers, estuaries) still
#   shows a strong positive MNDWI signal even when NDWI is weak.
# ================================

def compute_ndwi(image_hwc: np.ndarray) -> np.ndarray:
    """
    NDWI = (Green − NIR) / (Green + NIR + ε)
    image_hwc : (H, W, ≥8)  float32 in [0,1]
    returns   : (H, W, 1)   in [−1, 1]
    Best for  : clear open water (lakes, large rivers, ocean)
    """
    green = image_hwc[:, :, 2]
    nir   = image_hwc[:, :, 7]
    ndwi  = (green - nir) / (green + nir + 1e-6)
    return ndwi[:, :, np.newaxis]


def compute_mndwi(image_hwc: np.ndarray) -> np.ndarray:
    """
    MNDWI = (Green − SWIR1) / (Green + SWIR1 + ε)
    image_hwc : (H, W, ≥11)  float32 in [0,1]
    returns   : (H, W, 1)    in [−1, 1]
    Best for  : turbid water, braided/glacial rivers, urban water,
                coastal estuaries — cases where NDWI underperforms.
    """
    green = image_hwc[:, :, 2]
    swir1 = image_hwc[:, :, 10]   # B11 = SWIR1
    mndwi = (green - swir1) / (green + swir1 + 1e-6)
    return mndwi[:, :, np.newaxis]


def build_input(image_hwc: np.ndarray) -> np.ndarray:
    """
    Concatenate 12 spectral bands + NDWI + MNDWI → (H, W, 14).
    Call this instead of manually stacking in the dataset.
    """
    bands  = image_hwc[:, :, :12]            # (H, W, 12)
    ndwi   = compute_ndwi(image_hwc)         # (H, W,  1)
    mndwi  = compute_mndwi(image_hwc)        # (H, W,  1)
    return np.concatenate([bands, ndwi, mndwi], axis=2)   # (H, W, 14)


# ================================
# DATASET
# ================================
class SWEDDataset(Dataset):
    """
    Handles SWED_real .npy tile pairs.
    Naming convention:  <prefix>_image_<suffix>.npy  ↔  <prefix>_chip_<suffix>.npy
    Falls back to exact-name matching if the above does not find pairs.
    """
    def __init__(self, img_dir, mask_dir, img_size=256, augment=False, indices=None):
        self.img_dir  = img_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.augment  = augment

        all_images = sorted([f for f in os.listdir(img_dir) if f.endswith(".npy")])
        mask_set   = set(os.listdir(mask_dir))

        # Primary naming: _image_ ↔ _chip_
        pairs = [
            (f, f.replace("_image_", "_chip_"))
            for f in all_images
            if f.replace("_image_", "_chip_") in mask_set
        ]

        # Fallback: same filename in both dirs
        if not pairs:
            pairs = [(f, f) for f in all_images if f in mask_set]

        if indices is not None:
            pairs = [pairs[i] for i in indices]

        self.pairs = pairs
        print(f"  SWEDDataset: {len(self.pairs)} pairs  augment={augment}")

    def __len__(self):
        return len(self.pairs)

    # ── water-pixel fraction (for weighted sampler) ──────────────────────────
    def water_fraction(self, idx):
        _, mask_name = self.pairs[idx]
        mask = np.load(os.path.join(self.mask_dir, mask_name))
        if mask.ndim == 3:
            mask = mask[0]
        return float((mask > 0).mean())

    def __getitem__(self, idx):
        img_name, mask_name = self.pairs[idx]

        image = np.load(os.path.join(self.img_dir,  img_name))
        mask  = np.load(os.path.join(self.mask_dir, mask_name))

        # Layout normalisation → (H, W, C) / (H, W)
        if image.ndim == 3 and image.shape[0] <= 13:
            image = image.transpose(1, 2, 0)
        if mask.ndim == 3:
            mask = mask[0]

        image = image.astype(np.float32)
        mask  = mask.astype(np.float32)

        # Normalise image to [0, 1]
        image = np.clip(image, 0, 3000) / 3000.0

        # Binarise mask
        if mask.max() > 1:
            mask = (mask > 0).astype(np.float32)

        # Resize if needed
        if image.shape[0] != self.img_size or image.shape[1] != self.img_size:
            from skimage.transform import resize
            image = resize(image, (self.img_size, self.img_size, image.shape[2]),
                           anti_aliasing=True).astype(np.float32)
            mask  = resize(mask,  (self.img_size, self.img_size),
                           anti_aliasing=False, order=0).astype(np.float32)

        # ★ Build 14-channel input: 12 spectral + NDWI + MNDWI
        image = build_input(image)   # (H, W, 14)

        # ── Augmentation ──────────────────────────────────────────────────────
        if self.augment:
            # Geometric
            if np.random.rand() > 0.5:
                image = np.fliplr(image).copy(); mask = np.fliplr(mask).copy()
            if np.random.rand() > 0.5:
                image = np.flipud(image).copy(); mask = np.flipud(mask).copy()
            k = np.random.randint(0, 4)
            image = np.rot90(image, k).copy(); mask = np.rot90(mask, k).copy()

            # Brightness jitter (spectral bands only, not NDWI/MNDWI channels)
            if np.random.rand() > 0.5:
                image[:, :, :12] = np.clip(
                    image[:, :, :12] * np.random.uniform(0.75, 1.25), 0, 1)

            # Channel-wise jitter (spectral bands only)
            if np.random.rand() > 0.4:
                factors = np.random.uniform(0.85, 1.15, size=(1, 1, 12)).astype(np.float32)
                image[:, :, :12] = np.clip(image[:, :, :12] * factors, 0, 1)

            # Gaussian noise (spectral bands only)
            if np.random.rand() > 0.5:
                noise = np.random.normal(0, 0.008, image[:, :, :12].shape).astype(np.float32)
                image[:, :, :12] = np.clip(image[:, :, :12] + noise, 0, 1)

            # Random crop → resize back (32–96 px window)
            if np.random.rand() > 0.4:
                from skimage.transform import resize as sk_resize
                crop = np.random.randint(32, 96)
                x = np.random.randint(0, self.img_size - crop)
                y = np.random.randint(0, self.img_size - crop)
                image = sk_resize(image[y:y+crop, x:x+crop, :],
                                  (self.img_size, self.img_size, image.shape[2]),
                                  anti_aliasing=True).astype(np.float32)
                mask  = sk_resize(mask[y:y+crop, x:x+crop],
                                  (self.img_size, self.img_size),
                                  anti_aliasing=False, order=0).astype(np.float32)

            # ★ Re-compute NDWI/MNDWI after augmentation so index channels
            #   remain consistent with the (possibly jittered) spectral bands.
            image[:, :, 12:13] = compute_ndwi(image)
            image[:, :, 13:14] = compute_mndwi(image)

        image = torch.from_numpy(image.transpose(2, 0, 1))   # (14, H, W)
        mask  = torch.from_numpy(mask).unsqueeze(0)           # ( 1, H, W)
        return image, mask


# ================================
# VERIFY
# ================================
def verify_dataset(img_dir, mask_dir):
    ds = SWEDDataset(img_dir, mask_dir, augment=False)
    img, mask = ds[0]
    print(f"=== Dataset verification ===")
    print(f"  Image : {img.shape}  min={img.min():.3f}  max={img.max():.3f}")
    print(f"  Mask  : {mask.shape}  unique={torch.unique(mask).tolist()}")
    print(f"  Water : {int(mask.sum())} / {mask.numel()} px")
    print(f"  Chan 12 (NDWI)  : min={img[12].min():.3f}  max={img[12].max():.3f}")
    print(f"  Chan 13 (MNDWI) : min={img[13].min():.3f}  max={img[13].max():.3f}")
    assert img.shape  == (IN_CHANNELS, IMG_SIZE, IMG_SIZE), \
        f"Expected ({IN_CHANNELS}, {IMG_SIZE}, {IMG_SIZE}), got {img.shape}"
    assert mask.shape == (1, IMG_SIZE, IMG_SIZE)
    assert mask.max() <= 1.0
    print("  All checks passed!\n")
    return ds

full_ds = verify_dataset(IMG_DIR, MASK_DIR)

N        = len(full_ds)
val_n    = int(N * VAL_SPLIT)
train_n  = N - val_n

gen = torch.Generator().manual_seed(SEED)
train_idx, val_idx = torch.utils.data.random_split(
    range(N), [train_n, val_n], generator=gen
)
train_idx = list(train_idx); val_idx = list(val_idx)

train_ds = SWEDDataset(IMG_DIR, MASK_DIR, img_size=IMG_SIZE, augment=True,  indices=train_idx)
val_ds   = SWEDDataset(IMG_DIR, MASK_DIR, img_size=IMG_SIZE, augment=False, indices=val_idx)

# ── Weighted sampler ──────────────────────────────────────────────────────────
print("Computing sample weights (this may take ~30 s on 25K tiles)…")
fracs   = np.array([train_ds.water_fraction(i) for i in range(len(train_ds))])
weights = fracs + 0.05
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {train_n}  |  Val: {val_n}  |  Steps/epoch: {len(train_loader)}\n")


# ================================
# MODEL
# ★ in_channels=14 to accept the extra MNDWI channel
# ================================
model = smp.create_model(
    arch            = ARCH,
    encoder_name    = ENCODER,
    encoder_weights = "imagenet",
    in_channels     = IN_CHANNELS,   # 14
    classes         = 1,
)
model = model.to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M\n")


# ================================
# LOSS  — Focal + Dice + BCE
# ================================
dice_loss  = smp.losses.DiceLoss(mode="binary")
focal_loss = smp.losses.FocalLoss(mode="binary", gamma=2.0, alpha=0.75)
bce_loss   = nn.BCEWithLogitsLoss()

def combined_loss(pred, target):
    return (0.25 * bce_loss(pred, target)
          + 0.35 * dice_loss(pred, target)
          + 0.40 * focal_loss(pred, target))


# ================================
# OPTIMIZER + LR SCHEDULE
# ================================
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    cosine   = 0.5 * (1 + np.cos(np.pi * progress))
    return LR_MIN / LR + (1 - LR_MIN / LR) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ================================
# METRICS
# ================================
def compute_metrics(pred_logits, target, threshold=0.5):
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    tp   = (pred * target).sum(dim=(1,2,3))
    fp   = (pred * (1 - target)).sum(dim=(1,2,3))
    fn   = ((1 - pred) * target).sum(dim=(1,2,3))
    iou  = (tp + 1e-6) / (tp + fp + fn + 1e-6)
    f1   = (2*tp + 1e-6) / (2*tp + fp + fn + 1e-6)
    return iou.mean().item(), f1.mean().item()


# ================================
# TRAINING LOOP
# ================================
best_iou = 0.0
history  = {"train_loss": [], "val_iou": [], "val_f1": [], "lr": []}

print("=" * 60)
print(f" Starting training  ({EPOCHS} epochs, accum={ACCUM_STEPS})")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, (images, masks) in enumerate(
        tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} train"), start=1
    ):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        preds = model(images)
        loss  = combined_loss(preds, masks) / ACCUM_STEPS
        loss.backward()

        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item() * ACCUM_STEPS

    avg_loss = epoch_loss / len(train_loader)
    history["train_loss"].append(avg_loss)
    history["lr"].append(optimizer.param_groups[0]["lr"])
    scheduler.step()

    # ── Validation ───────────────────────────────────────────────────────────
    model.eval()
    all_iou, all_f1 = [], []
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} val  "):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            preds = model(images)
            iou, f1 = compute_metrics(preds, masks)
            all_iou.append(iou); all_f1.append(f1)

    avg_iou = np.mean(all_iou)
    avg_f1  = np.mean(all_f1)
    history["val_iou"].append(avg_iou)
    history["val_f1"].append(avg_f1)
    cur_lr  = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch:02d} | Loss={avg_loss:.4f} | IoU={avg_iou:.4f} | "
          f"F1={avg_f1:.4f} | LR={cur_lr:.2e}")

    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ Saved best model  (IoU={best_iou:.4f})")

print(f"\nTraining complete.  Best Val IoU: {best_iou:.4f}")


# ================================
# TRAINING CURVES
# ================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], color="#D85A30"); axes[0].set_title("Train Loss")
axes[1].plot(history["val_iou"],  color="#1D9E75", label="IoU")
axes[1].plot(history["val_f1"],   color="#534AB7", label="F1", linestyle="--")
axes[1].set_title("Val Metrics"); axes[1].legend()
axes[2].plot(history["lr"], color="#E8922A"); axes[2].set_title("LR Schedule")
for ax in axes: ax.set_xlabel("Epoch")
plt.tight_layout()
plt.savefig("/home/jupyter-228w1a1286/dinesh-major/v3/training_curves_v3_1.png", dpi=150)
plt.show()
print("Curves saved.")


# ================================
# THRESHOLD SWEEP  (on val set)
# ================================
print("\n=== Threshold sweep ===")
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE, weights_only=True))
model.eval()

thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
best_thresh, best_thresh_iou = 0.5, 0.0

with torch.no_grad():
    all_logits, all_masks = [], []
    for images, masks in val_loader:
        logits = model(images.to(DEVICE)).cpu()
        all_logits.append(logits); all_masks.append(masks)
    all_logits = torch.cat(all_logits); all_masks = torch.cat(all_masks)

for t in thresholds:
    ious, f1s = [], []
    for i in range(0, len(all_logits), BATCH_SIZE):
        iou, f1 = compute_metrics(
            all_logits[i:i+BATCH_SIZE], all_masks[i:i+BATCH_SIZE], threshold=t)
        ious.append(iou); f1s.append(f1)
    miou = np.mean(ious)
    print(f"  t={t:.2f}  IoU={miou:.4f}  F1={np.mean(f1s):.4f}")
    if miou > best_thresh_iou:
        best_thresh_iou = miou; best_thresh = t

print(f"\nBest threshold: {best_thresh}  →  IoU={best_thresh_iou:.4f}")


# ================================
# INFERENCE ON .npy VAL TILES
# ================================
water_cmap = ListedColormap(["none", "#1a75ff"])
shown, SHOW_N, MIN_PX = 0, 8, 50

print(f"\nVisualising val tiles (threshold={best_thresh}) …\n")
model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        preds     = torch.sigmoid(model(images.to(DEVICE))).cpu()
        preds_bin = (preds > best_thresh).float().numpy()
        for i in range(images.shape[0]):
            gt = masks.numpy()[i, 0]; pr = preds_bin[i, 0]
            if int(gt.sum()) < MIN_PX: continue

            rgb  = np.clip(images.numpy()[i][[3,2,1]].transpose(1,2,0), 0, 1)
            inter = (pr * gt).sum(); union = pr.sum() + gt.sum() - inter
            iou_t = float((inter + 1e-6) / (union + 1e-6))
            prec  = float(((pr*gt).sum()+1e-6) / (pr.sum()+1e-6))
            rec   = float(((pr*gt).sum()+1e-6) / (gt.sum()+1e-6))

            fig, axes = plt.subplots(1, 4, figsize=(20, 4))
            axes[0].imshow(rgb)
            axes[0].set_title("RGB"); axes[0].axis("off")
            axes[1].imshow(gt, cmap="Blues", vmin=0, vmax=1)
            axes[1].set_title("Ground truth"); axes[1].axis("off")
            axes[2].imshow(pr, cmap="Blues", vmin=0, vmax=1)
            axes[2].set_title("Prediction"); axes[2].axis("off")
            axes[3].imshow(rgb)
            axes[3].imshow(pr, cmap=water_cmap, alpha=0.6)
            axes[3].set_title(f"Overlay  IoU={iou_t:.3f}\nP={prec:.2f}  R={rec:.2f}")
            axes[3].axis("off")
            plt.tight_layout(); plt.show()

            print(f"  [{shown+1}] gt={int(gt.sum())}px | pred={int(pr.sum())}px | "
                  f"IoU={iou_t:.3f} | P={prec:.2f} | R={rec:.2f}")
            shown += 1
            if shown >= SHOW_N: break
        if shown >= SHOW_N: break


# ================================
# TEST SET INFERENCE
# The test folder has 98 files but they are NOT .npy.
# This block auto-detects the format and runs inference.
# Supported: .tif / .tiff (rasterio), .png / .jpg (PIL)
# ================================
print("\n=== Test-set inference ===")
test_img_dir  = os.path.join(TEST_DIR, "images")
test_mask_dir = os.path.join(TEST_DIR, "labels")

test_files = sorted(os.listdir(test_img_dir))
print(f"  Found {len(test_files)} files in test/images")
print(f"  Extensions: {set(os.path.splitext(f)[1] for f in test_files)}")


def load_test_image(path):
    """Load a single test image regardless of format. Returns (H,W,C) float32 [0,1]."""
    ext = os.path.splitext(path)[1].lower()

    if ext in (".tif", ".tiff"):
        import rasterio
        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32)   # (C, H, W)
        arr = arr.transpose(1, 2, 0)
        return np.clip(arr, 0, 3000) / 3000.0

    elif ext in (".png", ".jpg", ".jpeg"):
        from PIL import Image
        img = Image.open(path).convert("RGB")
        arr = np.array(img, dtype=np.float32) / 255.0   # (H, W, 3)
        arr = np.concatenate([arr] * 4, axis=2)[:, :, :12]
        return arr

    elif ext == ".npy":
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 3 and arr.shape[0] <= 13:
            arr = arr.transpose(1, 2, 0)
        return np.clip(arr, 0, 3000) / 3000.0

    else:
        raise ValueError(f"Unsupported test file format: {ext}")


def load_test_mask(path):
    """Load a single test mask. Returns (H,W) float32 {0,1}."""
    if not os.path.exists(path):
        return None
    ext = os.path.splitext(path)[1].lower()
    if ext in (".tif", ".tiff"):
        import rasterio
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
        return (arr > 0).astype(np.float32)
    elif ext in (".png", ".jpg", ".jpeg"):
        from PIL import Image
        arr = np.array(Image.open(path).convert("L"), dtype=np.float32)
        return (arr > 127).astype(np.float32)
    elif ext == ".npy":
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 3: arr = arr[0]
        return (arr > 0).astype(np.float32)
    return None


from skimage.transform import resize as sk_resize

model.eval()
test_ious = []

with torch.no_grad():
    for fname in tqdm(test_files, desc="Test inference"):
        img_path = os.path.join(test_img_dir, fname)

        stem = os.path.splitext(fname)[0].replace("_image_", "_chip_")
        mask_path = None
        for ext in (".npy", ".tif", ".tiff", ".png", ".jpg"):
            candidate = os.path.join(test_mask_dir, stem + ext)
            if os.path.exists(candidate):
                mask_path = candidate; break

        try:
            image = load_test_image(img_path)
        except Exception as e:
            print(f"  [SKIP] {fname}: {e}"); continue

        # Resize to model input
        if image.shape[0] != IMG_SIZE or image.shape[1] != IMG_SIZE:
            image = sk_resize(image, (IMG_SIZE, IMG_SIZE, image.shape[2]),
                              anti_aliasing=True).astype(np.float32)

        # ★ Build 14-channel input using shared helper
        image  = build_input(image)   # (H, W, 14)
        tensor = torch.from_numpy(image.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)

        pred   = torch.sigmoid(model(tensor)).squeeze().cpu().numpy()
        pred_b = (pred > best_thresh).astype(np.float32)

        if mask_path:
            gt = load_test_mask(mask_path)
            if gt is not None:
                if gt.shape != (IMG_SIZE, IMG_SIZE):
                    gt = sk_resize(gt, (IMG_SIZE, IMG_SIZE),
                                   anti_aliasing=False, order=0).astype(np.float32)
                inter = (pred_b * gt).sum(); union = pred_b.sum() + gt.sum() - inter
                iou_t = float((inter + 1e-6) / (union + 1e-6))
                test_ious.append(iou_t)

if test_ious:
    print(f"\nTest IoU (n={len(test_ious)}): mean={np.mean(test_ious):.4f}  "
          f"median={np.median(test_ious):.4f}  min={np.min(test_ious):.4f}")
else:
    print("No matched masks found in test set — IoU not computed.")
    print("Predictions generated. Add mask files to test/labels to get IoU.")

print(f"\nAll done!  Best Val IoU: {best_thresh_iou:.4f} @ t={best_thresh}")